# Roomor · DistilBERT sul Model 1 (temi) · controllo di stabilità con altri seed

**Cosa fa:** rifà l'addestramento a 8 epoche con **seed 1** e poi con **seed 2**. Tutto il resto è uguale al quaderno `distilbert_topics_8epochs.ipynb`: cambiano solo il seed e i nomi dei file.

**Perché:** il seed decide le parti casuali (pesi iniziali dello strato finale, ordine delle frasi, dropout). Se il buon risultato ottenuto con il seed 42 dipendesse dalla fortuna, con altri seed sparirebbe. La divisione train / validation / test **non** cambia: è sempre quella con random_state 42.

**Regola decisa prima di partire:** teniamo DistilBERT solo se **tutti e tre i seed** (42, 1, 2) battono micro F1 0.857 e macro F1 0.852 sul test.

**Da dichiarare:** il test era già stato guardato due volte (3 epoche e 8 epoche, seed 42). Con i due nuovi seed lo si calcola altre due volte, ma solo per applicare la regola: non si sceglie il seed migliore.

**Cosa esce, per ogni seed:** `distilbert8_seed1_val_probs.csv`, `distilbert8_seed1_test_probs.csv`, `distilbert8_seed1_epochs.csv` (con `seed2` per il secondo) e il modello su Google Drive.

**Come si usa:** attiva la GPU, poi esegui le celle in ordine dall'alto in basso e aspetta che ognuna finisca.

## 0. Interruttori: prova veloce e seed

- `SEED = 1` per il primo addestramento, `SEED = 2` per il secondo. Se scrivi un altro numero la cella si ferma.
- `QUICK_TEST = True`: usa solo 300 frasi di train, 100 di validation e 100 di test, e 2 epoche. Serve **solo** a controllare che tutte le celle funzionino. **I numeri che escono non valgono nulla.**
- `QUICK_TEST = False`: addestramento vero, con tutte le frasi e 8 epoche.

La prova veloce salva in una cartella diversa di Drive (`distilbert_topics_8epochs_seed1_QUICKTEST`) e scarica file che iniziano con `QUICKTEST_`: così non si possono confondere con i risultati veri.

**Quando cambi il valore**, usa il menu **Runtime → Riavvia sessione ed esegui tutto**: non basta rieseguire questa cella.

In [ ]:
QUICK_TEST = True          # True = quick check of the notebook; False = real training
SEED = 1                   # first run: 1, second run: 2 (seed 42 is already done)
assert SEED in (1, 2), "SEED must be 1 or 2"

## 1. Installare le librerie e controllare la GPU

- **transformers** è la libreria che contiene DistilBERT. **torch** (PyTorch), che fa i calcoli, è già presente su Colab.
- La **GPU** è un processore che fa moltissimi calcoli in parallelo: l'addestramento diventa molto più veloce che con il processore normale.

`assert` ferma tutto se la GPU non è attiva. Le versioni stampate vanno annotate, perché servono per dichiarare come abbiamo ottenuto i risultati.

In [ ]:
%pip install -q transformers

import random
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

assert torch.cuda.is_available(), "GPU not active: Runtime > Change runtime type > T4 GPU"
device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "- transformers", transformers.__version__)

## 2. Le scelte dell'esperimento

Tutte le scelte stanno qui, così si vedono e si dichiarano facilmente. Sono fissate **prima** di vedere i risultati.

- **Epoca** (`EPOCHS = 8`): un giro completo su tutte le 16.173 frasi di train. Con 8 epoche il modello vede ogni frase fino a 8 volte. Troppo poche: impara poco. Troppe: impara a memoria le frasi di train e peggiora su quelle nuove (overfitting).
- **Learning rate** (`2e-5` = 0,00002): quanto è grande ogni correzione dei pesi. Troppo grande: rovina ciò che il modello già sa. Troppo piccolo: non impara quasi nulla. Per il fine-tuning si usano valori molto piccoli.
- **Batch** (`32`): il modello legge 32 frasi alla volta e poi si corregge.
- **Lunghezza massima** (`64` token): le frasi più lunghe vengono tagliate. Più sotto contiamo quante sono.
- **Seed** (scelto nella sezione 0): fissa le scelte casuali (pesi iniziali dell'ultimo strato, ordine delle frasi), così rifacendo l'esperimento i numeri sono molto simili. Sulla GPU possono restare piccolissime differenze.

`TOPICS` ha lo stesso ordine di `topic_model.py`.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
EPOCHS = 2 if QUICK_TEST else 8
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
MAX_TOKENS = 64
PREFIX = "QUICKTEST_" if QUICK_TEST else ""          # start of the names of the result files
print("QUICK TEST: the results are NOT valid" if QUICK_TEST else "REAL TRAINING", "- seed", SEED)

TOPICS = [
    "Clean", "Comfort", "Facilities/Amenities", "Location", "Restaurant (dinner)",
    "Staff", "View (Balcony)", "Breakfast", "Room", "Pool", "Beach",
    "Bathroom/Shower (toilet)", "Bar", "Bed", "Parking", "Noise",
    "Reception-checkin", "Lift", "Value for money", "Wi-Fi", "Generic",
]

def fix_seed():
    """Fix all the random choices (Python, numpy, torch on CPU and GPU)."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)            # it fixes the GPU too
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fix_seed()

## 3. Collegare Google Drive

Il computer di Colab è **temporaneo**: quando si chiude la sessione, i file spariscono. Per questo colleghiamo Google Drive. Lo facciamo **prima** dell'addestramento, così se qualcosa non va non perdiamo il lavoro.

Si apre una finestra che chiede il permesso: scegli il tuo account e accetta. I risultati finiscono nella cartella `Roomor/distilbert_topics_8epochs_seed1` (o `seed2`) del tuo Drive (con `_QUICKTEST` alla fine per la prova veloce).

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")

DRIVE_FOLDER = Path("/content/drive/MyDrive/Roomor") / (f"distilbert_topics_8epochs_seed{SEED}" + ("_QUICKTEST" if QUICK_TEST else ""))
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
print("Results will be saved in", DRIVE_FOLDER)

## 4. Caricare i tre file

Compare il pulsante **Scegli file**: seleziona insieme `hrast_train.csv`, `hrast_val.csv` e `hrast_test.csv` (sul tuo PC sono nella cartella `data/` della repo). Se i file sono già stati caricati, la cella non li chiede di nuovo.

- `keep_default_na=False`: una frase come "None" resta un testo e non diventa un valore vuoto.
- Gli `assert` controllano che i numeri siano 16.173 / 3.466 / 3.466 e che i temi siano nello stesso ordine: altrimenti il confronto con il Model 1 non sarebbe corretto.
- Se la prova veloce è attiva, dopo i controlli tiene solo le prime 300 / 100 / 100 frasi.

In [ ]:
NAMES = ["hrast_train.csv", "hrast_val.csv", "hrast_test.csv"]

if not all(Path(name).exists() for name in NAMES):
    files.upload()                     # shows the "Choose files" button

train_df, val_df, test_df = [pd.read_csv(name, keep_default_na=False) for name in NAMES]

assert (len(train_df), len(val_df), len(test_df)) == (16173, 3466, 3466), "Wrong number of sentences!"
for table in (train_df, val_df, test_df):
    assert list(table.columns) == ["review"] + TOPICS, "Wrong columns!"

if QUICK_TEST:                         # keep only a few sentences, to check that everything works
    train_df, val_df, test_df = train_df.head(300), val_df.head(100), test_df.head(100)
print("Train:", len(train_df), "- Validation:", len(val_df), "- Test:", len(test_df))

## 5. Il tokenizer

Un modello fa calcoli: non legge lettere, ma **numeri**. Il **tokenizer** divide la frase in pezzi (**token**) e dà a ognuno un numero. Una parola comune è un solo token; una parola rara viene spezzata in più pezzi, così il modello può leggere anche parole che non conosce.

- Aggiunge due token speciali: `[CLS]` all'inizio e `[SEP]` alla fine.
- **uncased** = tutto minuscolo: "Staff" e "staff" sono la stessa cosa.
- Deve essere **lo stesso** con cui DistilBERT è stato pre-addestrato, altrimenti i numeri non corrisponderebbero alle parole giuste.

La cella mostra un esempio e conta le frasi più lunghe di 64 token. (Se compare un avviso su `HF_TOKEN`, si può ignorare: il modello è pubblico.)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = "The soundproofing was terrible but the staff were lovely"
print(tokenizer.convert_ids_to_tokens(tokenizer(example)["input_ids"]))

for name, table in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    lengths = [len(ids) for ids in tokenizer(list(table["review"]))["input_ids"]]
    n_long = sum(length > MAX_TOKENS for length in lengths)
    print(f"{name}: longest {max(lengths)} tokens, cut sentences {n_long} ({n_long / len(table):.2%})")

## 6. Preparare i gruppi di frasi (batch)

- `make_batch` prende un gruppo di righe, trasforma i testi in token (`truncation` taglia a 64) e aggiunge dei riempitivi (`padding`) per portare tutte le frasi del gruppo alla stessa lunghezza. La **attention mask** dice al modello quali token sono riempitivi da ignorare.
- Le risposte corrette diventano numeri decimali (0.0 / 1.0), perché la formula dell'errore lo richiede.
- `shuffle=True` solo nel train: a ogni epoca le frasi arrivano in un ordine diverso. Validation e test **non** vengono mescolati: ogni probabilità deve restare accanto alla sua frase.

In [ ]:
def make_batch(rows):
    """From a list of (text, 21 labels) to the tensors that the model needs."""
    texts = [text for text, labels in rows]
    batch = tokenizer(texts, truncation=True, max_length=MAX_TOKENS, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor(np.array([labels for text, labels in rows]), dtype=torch.float)
    return batch

def make_loader(table, shuffle):
    """Split a table in batches of 32 sentences."""
    rows = list(zip(table["review"], table[TOPICS].values))
    return DataLoader(rows, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=make_batch)

fix_seed()
train_loader = make_loader(train_df, shuffle=True)
val_loader = make_loader(val_df, shuffle=False)
test_loader = make_loader(test_df, shuffle=False)
print("Batches per epoch:", len(train_loader))

## 7. Caricare DistilBERT: pre-addestramento e fine-tuning

- **Pre-addestrato:** DistilBERT ha già imparato l'inglese da moltissimi testi (Wikipedia e libri), imparando a indovinare parole nascoste. È una versione più piccola e veloce di BERT. Sa già che "staff" e "personnel" si usano in contesti simili, e tiene conto dell'ordine delle parole: il TF-IDF non può.
- **Fine-tuning:** partiamo da ciò che sa e correggiamo un po' i suoi pesi sul nostro compito. Da zero servirebbero enormemente più testi delle nostre 16.173 frasi.
- **21 uscite:** aggiungiamo in cima uno strato nuovo con un numero per tema. Parte da valori casuali: per questo può comparire un avviso che dice che alcuni pesi sono nuovi ("newly initialized"). È normale.

### Perché la sigmoide e non la softmax?

Il modello produce 21 punteggi senza limiti, che vanno trasformati in probabilità tra 0 e 1.

- La **softmax** fa sommare a 1 le 21 probabilità: i temi si fanno concorrenza, quindi serve quando la risposta giusta è **una sola**.
- La **sigmoide** trasforma ogni punteggio da solo: "la camera era pulita ma cara" può avere Clean 0,95, Room 0,90 e Value for money 0,85 insieme. Da noi una frase può avere più temi (**multi-etichetta**), quindi serve la sigmoide.

È come avere 21 domande sì/no, proprio come le 21 regressioni logistiche del Model 1. `problem_type="multi_label_classification"` dice al modello di calcolare l'errore così (*binary cross-entropy*, una per tema).

In [ ]:
fix_seed()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TOPICS),
    problem_type="multi_label_classification",
    id2label={i: topic for i, topic in enumerate(TOPICS)},      # output number -> topic name
    label2id={topic: i for i, topic in enumerate(TOPICS)},
)
model = model.to(device)               # move the model to the GPU
print("Weights to train:", sum(p.numel() for p in model.parameters()))

## 8. Due funzioni: le probabilità e gli F1 sul validation

- `predict_probabilities`: per ogni frase il modello dà 21 punteggi (`logits`), la **sigmoide** li trasforma in probabilità tra 0 e 1. `torch.no_grad()` = calcola senza imparare. `model.eval()` = modalità valutazione (il *dropout* è spento).
- `validation_scores`: per ogni tema prova le 18 soglie 0,05, 0,10, ... 0,90 e tiene quella con l'F1 più alto, come per il Model 1; poi calcola micro e macro F1. Le soglie sono scelte e misurate sullo **stesso** validation, quindi questi numeri sono un po' ottimisti: servono solo a confrontare le epoche tra loro.

In [ ]:
def predict_probabilities(loader, the_model):
    """One row per sentence, one column per topic: probabilities between 0 and 1."""
    the_model.eval()
    parts = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = the_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            parts.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(parts)

CANDIDATES = np.round(np.arange(1, 19) * 0.05, 2)      # 0.05, 0.10, ..., 0.90

def validation_scores(probs):
    """Choose one threshold per topic on validation, then micro F1 and macro F1 on validation."""
    Y = val_df[TOPICS].values
    pred = np.zeros(Y.shape, dtype=int)
    for j in range(len(TOPICS)):
        best_f1 = -1
        for t in CANDIDATES:
            guess = (probs[:, j] >= t).astype(int)
            f1 = f1_score(Y[:, j], guess, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                pred[:, j] = guess
    return (f1_score(Y, pred, average="micro", zero_division=0),
            f1_score(Y, pred, average="macro", zero_division=0))

## 8b. Addestrare e tenere l'epoca migliore

Per ogni gruppo di 32 frasi:

1. il modello fa le sue previsioni e calcola l'errore (**loss**): `model(**batch).loss`;
2. `loss.backward()` calcola in che direzione spostare ogni peso per sbagliare meno;
3. `clip_grad_norm_` limita le correzioni troppo grandi (una sicurezza);
4. `optimizer.step()` sposta i pesi; quanto, lo decide il learning rate;
5. `scheduler.step()` abbassa un po' il learning rate: parte da 2e-5 e arriva a 0 alla fine delle 8 epoche. Per questo l'epoca 3 di qui **non** è uguale all'epoca 3 del primo quaderno;
6. `optimizer.zero_grad()` cancella i calcoli prima del gruppo successivo.

Alla fine di ogni epoca stampa la loss del train, la loss e gli F1 del **validation**. Se il macro F1 è il più alto finora, copia i pesi del modello (`best_weights`). Alla fine rimette nel modello i pesi dell'epoca migliore. Il **test non viene mai usato** qui.

`best_macro = -1` fa accettare sempre la prima epoca; il segno `>` (e non `>=`) fa vincere la prima in caso di parità. La validation loss è calcolata dalle probabilità con la stessa formula (*binary cross-entropy*).

**Come leggerla:** se la loss di train scende ma quella di validation sale, il modello sta imparando a memoria. Può succedere nelle ultime epoche: per questo teniamo la migliore.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
n_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=n_steps)

Y_val = val_df[TOPICS].values
best_macro, best_epoch, best_weights = -1, None, None
history = []

fix_seed()
start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total, n = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device)
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total += loss.item() * len(batch["labels"])
        n += len(batch["labels"])

    # End of the epoch: how good is the model on the validation set?
    probs = predict_probabilities(val_loader, model)
    safe = np.clip(probs.astype(float), 1e-7, 1 - 1e-7)      # avoids the logarithm of 0
    val_loss = -(Y_val * np.log(safe) + (1 - Y_val) * np.log(1 - safe)).mean()
    micro, macro = validation_scores(probs)
    history.append([epoch, total / n, val_loss, micro, macro])
    print(f"Epoch {epoch}: train loss {total / n:.4f} - validation loss {val_loss:.4f}"
          f" - validation micro F1 {micro:.4f} - macro F1 {macro:.4f}"
          f" - {(time.time() - start) / 60:.1f} minutes")

    if macro > best_macro:             # best epoch so far: keep a copy of the weights
        best_macro, best_epoch = macro, epoch
        best_weights = {name: w.detach().cpu().clone() for name, w in model.state_dict().items()}

model.load_state_dict(best_weights)    # back to the weights of the best epoch
history = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "val_micro_f1", "val_macro_f1"])
history.to_csv(DRIVE_FOLDER / f"{PREFIX}distilbert8_seed{SEED}_epochs.csv", index=False)
print("Best epoch:", best_epoch, "- validation macro F1", round(best_macro, 4))

## 9. Salvare il modello su Google Drive

Salviamo i pesi dell'epoca migliore. Lo facciamo **subito**, prima di tutto il resto: se la sessione si chiude, l'addestramento non è perso. Salviamo anche il tokenizer, perché servono insieme per usare il modello sulle recensioni del 515K.

Non caricare il modello su GitHub: pesa circa 250 MB ed è stato addestrato su HRAST (uso non commerciale).

In [ ]:
model.save_pretrained(DRIVE_FOLDER / "model")
tokenizer.save_pretrained(DRIVE_FOLDER / "model")
for path in sorted((DRIVE_FOLDER / "model").iterdir()):
    print(f"{path.name:<28}{path.stat().st_size / 1e6:>8.1f} MB")

## 10. Calcolare le probabilità su validation e test

Usiamo i pesi dell'**epoca migliore**. Otteniamo una tabella con una riga per frase e una colonna per tema, più il testo in `review` per controllare l'abbinamento. Qui non confrontiamo il test con le risposte corrette.

I file si salvano su Drive e poi si scaricano. Il browser può chiedere il permesso di scaricare più file: accetta.

In [ ]:
for name, table, loader in [("val", val_df, val_loader), ("test", test_df, test_loader)]:
    probs = pd.DataFrame(predict_probabilities(loader, model), columns=TOPICS)
    probs.insert(0, "review", table["review"].values)       # the text, to check the order later
    assert probs.shape == (len(table), 22) and probs[TOPICS].min().min() >= 0 and probs[TOPICS].max().max() <= 1
    probs.to_csv(DRIVE_FOLDER / f"{PREFIX}distilbert8_seed{SEED}_{name}_probs.csv", index=False, encoding="utf-8")
    print(f"{PREFIX}distilbert8_seed{SEED}_{name}_probs.csv:", probs.shape)

## 11. Controllare il modello salvato e scaricare i file

Ricarichiamo il modello da Drive e rifacciamo le previsioni sul validation: devono essere uguali (tranne differenze minuscole di arrotondamento). Così siamo sicuri che il modello salvato sia quello giusto. Poi proviamo una frase inventata e scarichiamo i tre file.

In [ ]:
saved_model = AutoModelForSequenceClassification.from_pretrained(DRIVE_FOLDER / "model").to(device)
saved_probs = pd.read_csv(DRIVE_FOLDER / f"{PREFIX}distilbert8_seed{SEED}_val_probs.csv", keep_default_na=False)
difference = np.abs(predict_probabilities(val_loader, saved_model) - saved_probs[TOPICS].values).max()
print("Maximum difference with the saved model:", difference)
assert difference < 0.001, "The saved model gives different predictions!"

example = tokenizer("The room was tiny but the staff were lovely", return_tensors="pt").to(device)
with torch.no_grad():
    example_probs = torch.sigmoid(saved_model(**example).logits)[0].cpu().numpy()
for i in example_probs.argsort()[::-1][:4]:            # the 4 topics with the highest probability
    print(f"  {TOPICS[i]:<28}{example_probs[i]:.3f}")

files.download(str(DRIVE_FOLDER / f"{PREFIX}distilbert8_seed{SEED}_val_probs.csv"))
files.download(str(DRIVE_FOLDER / f"{PREFIX}distilbert8_seed{SEED}_test_probs.csv"))
files.download(str(DRIVE_FOLDER / f"{PREFIX}distilbert8_seed{SEED}_epochs.csv"))

## Finito

- Se era la prova veloce (`QUICK_TEST = True`): cancella i file `QUICKTEST_`, metti `QUICK_TEST = False` nella cella 0 e usa **Runtime → Riavvia sessione ed esegui tutto**.
- Se era l'addestramento vero: metti i tre file scaricati nella cartella `data/` della repo (non vanno su GitHub).
- Annota: versioni delle librerie (sezione 1); le righe di ogni epoca (sezione 8b) sono anche in `distilbert8_seed1_epochs.csv` (o `seed2`).
- Dopo il seed 1: metti `SEED = 2` nella cella 0, lascia `QUICK_TEST = False` e usa di nuovo **Runtime → Riavvia sessione ed esegui tutto**.
- Alla fine dei due: **Runtime → Disconnetti ed elimina runtime**, per liberare la GPU.